In [2]:
import urllib.request
import json
import openai

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"
client = openai.OpenAI()


In [3]:
def get_popular_movies():
    req = urllib.request.Request(       # URL을 바로 넘기지 않고 Request 객체를 만듦
        f"{BASE_URL}/movies",
        headers={"User-Agent": "Mozilla/5.0"}  # 브라우저인 척하는 헤더 추가
    )
    with urllib.request.urlopen(req) as res:   # URL 대신 req 객체를 넘김
        return json.loads(res.read())
   # with urllib.request.urlopen(f"{BASE_URL}/movies") as res:
       # return json.loads(res.read())
       

def get_movie_details(id):
    req = urllib.request.Request(       # URL을 바로 넘기지 않고 Request 객체를 만듦
        f"{BASE_URL}/movies/{id}",
        headers={"User-Agent": "Mozilla/5.0"}  # 브라우저인 척하는 헤더 추가
    )
    with urllib.request.urlopen(req) as res:   # URL 대신 req 객체를 넘김
        return json.loads(res.read())
    # with urllib.request.urlopen(f"{BASE_URL}/movies/{id}") as res:
        # return json.loads(res.read())
    
def get_similar_movies(id):
    req = urllib.request.Request(       # URL을 바로 넘기지 않고 Request 객체를 만듦
        f"{BASE_URL}/movies/{id}/similar",
        headers={"User-Agent": "Mozilla/5.0"}  # 브라우저인 척하는 헤더 추가
    )
    with urllib.request.urlopen(req) as res:   # URL 대신 req 객체를 넘김
        return json.loads(res.read())
    # with urllib.request.urlopen(f"{BASE_URL}/movies/{id}/similar") as res:
        # return json.loads(res.read())

In [5]:
func_map = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details, 
    "get_similar_movies": get_similar_movies,
}

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get movies similar to a specific movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [7]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful movie assistant. Answer in Korean."
    }
]

In [8]:
def run_agent(user_input):
    messages.append({
        "role": "user",
        "content": user_input
    })
    print(f"User: {user_input}")

    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message

        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f"[{func_name}({func_args}) 호출 중...]")

                result = func_map[func_name](**func_args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
        else:
            print(f"Agent: {message.content}")
            break



In [ ]:
while True:
    user_input = input("User: ")
    if user_input in ("quit", "q"):
        break
    run_agent(user_input)



User: 
Agent: 안녕하세요! 어떤 정보를 찾고 계신가요? 영화에 대한 질문이나 추천이 필요하시면 말씀해 주세요.
